## **1. Load the feature table and look at the three parts.**

In [2]:
import pandas as pd

df = pd.read_parquet("../data/features.parquet")

print(df.shape)
print(df.groupby("part")["is_fraud"].agg(["count", "sum"]))
print(df.groupby("part")["time"].agg(["min", "max"]))

(1852394, 29)
         count   sum
part                
test    555719  2145
train  1296675  7506
                      min                 max
part                                         
test  2020-06-21 12:14:25 2020-12-31 23:59:34
train 2019-01-01 00:00:18 2020-06-21 12:13:37


##  **2. Choose the split dates**

In [3]:
cutoff = pd.Timestamp("2020-03-21")

train_all = df[df["part"] == "train"]
fit = train_all[train_all["time"] < cutoff]
val = train_all[train_all["time"] >= cutoff]

print("learning set:  ", len(fit), "payments,", fit["is_fraud"].sum(), "fraud")
print("validation set:", len(val), "payments,", val["is_fraud"].sum(), "fraud")

learning set:   1070966 payments, 6250 fraud
validation set: 225709 payments, 1256 fraud


## **3. Make the three sets: inputs (X) and answer key (y)**

In [4]:
import sys
sys.path.append("../src")
from features import FEATURE_COLUMNS

test_all = df[df["part"] == "test"]

X_fit, y_fit = fit[FEATURE_COLUMNS], fit["is_fraud"]
X_val, y_val = val[FEATURE_COLUMNS], val["is_fraud"]
X_test, y_test = test_all[FEATURE_COLUMNS], test_all["is_fraud"]

print(len(FEATURE_COLUMNS), "features")
print("learning:  ", X_fit.shape, y_fit.shape)
print("validation:", X_val.shape, y_val.shape)
print("replay:    ", X_test.shape, y_test.shape)

24 features
learning:   (1070966, 24) (1070966,)
validation: (225709, 24) (225709,)
replay:     (555719, 24) (555719,)


## **4. Check the split**

In [5]:
print("learning:  ", fit["time"].min(), "to", fit["time"].max(), "|", int(y_fit.sum()), "fraud")
print("validation:", val["time"].min(), "to", val["time"].max(), "|", int(y_val.sum()), "fraud")
print("replay:    ", test_all["time"].min(), "to", test_all["time"].max(), "|", int(y_test.sum()), "fraud")

print("learning ends before validation starts:", fit["time"].max() < val["time"].min())
print("validation ends before replay starts:  ", val["time"].max() < test_all["time"].min())

learning:   2019-01-01 00:00:18 to 2020-03-20 23:58:44 | 6250 fraud
validation: 2020-03-21 00:00:20 to 2020-06-21 12:13:37 | 1256 fraud
replay:     2020-06-21 12:14:25 to 2020-12-31 23:59:34 | 2145 fraud
learning ends before validation starts: True
validation ends before replay starts:   True


## **Phase 4 summary**
- Learning set: Jan 2019 to 20 Mar 2020, 1,070,966 payments, 6,250 fraud
- Validation set: 21 Mar to 21 Jun 2020, 225,709 payments, 1,256 fraud
- Replay set: 21 Jun to 31 Dec 2020, 555,719 payments, 2,145 fraud
- Split by date, never at random. The replay set is not touched until the final grading.

## **5.1 Baseline model: logistic regression**

In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

baseline = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=500))
baseline.fit(X_fit, y_fit)

base_scores = baseline.predict_proba(X_val)[:, 1]
print("baseline PR-AUC on validation:", round(average_precision_score(y_val, base_scores), 3))
print("random guessing would score about:", round(y_val.mean(), 4))

baseline PR-AUC on validation: 0.291
random guessing would score about: 0.0056


## **5.2 XGBoost model**

In [7]:
from xgboost import XGBClassifier

neg, pos = (y_fit == 0).sum(), (y_fit == 1).sum()

xgb = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=neg / pos, tree_method="hist",
    n_jobs=-1, random_state=1,
)
xgb.fit(X_fit, y_fit)

xgb_scores = xgb.predict_proba(X_val)[:, 1]
print("XGBoost PR-AUC on validation:", round(average_precision_score(y_val, xgb_scores), 3))

XGBoost PR-AUC on validation: 0.913


## **5.3 Compare: how much fraud do they catch?**

In [8]:
import numpy as np

yv = y_val.values

def caught_at_false_alarm(scores, rate):
    cutoff = np.quantile(scores[yv == 0], 1 - rate)
    flagged = scores >= cutoff
    return flagged[yv == 1].mean()

for name, s in [("baseline", base_scores), ("xgboost", xgb_scores)]:
    for rate in [0.001, 0.005, 0.01]:
        caught = caught_at_false_alarm(s, rate)
        print(f"{name:9s} | wrongly flagged honest payments: {rate*100:.1f}% | fraud caught: {caught*100:.1f}%")

baseline  | wrongly flagged honest payments: 0.1% | fraud caught: 7.0%
baseline  | wrongly flagged honest payments: 0.5% | fraud caught: 57.0%
baseline  | wrongly flagged honest payments: 1.0% | fraud caught: 71.4%
xgboost   | wrongly flagged honest payments: 0.1% | fraud caught: 83.8%
xgboost   | wrongly flagged honest payments: 0.5% | fraud caught: 95.3%
xgboost   | wrongly flagged honest payments: 1.0% | fraud caught: 97.8%


##  **5.4 Which features mattered most**

In [9]:
import pandas as pd

importance = pd.Series(xgb.feature_importances_, index=FEATURE_COLUMNS).sort_values(ascending=False)
print(importance.head(10).round(3))

is_night             0.232
amt                  0.142
cat_gas_transport    0.094
cat_grocery_pos      0.055
cat_food_dining      0.055
cat_shopping_net     0.046
cat_shopping_pos     0.042
cat_home             0.039
cat_entertainment    0.039
cat_grocery_net      0.038
dtype: float32


## **5.5 Save the model**

In [10]:
import os, joblib

os.makedirs("../models", exist_ok=True)
joblib.dump(xgb, "../models/fraud_model.joblib")

loaded = joblib.load("../models/fraud_model.joblib")
same = np.allclose(loaded.predict_proba(X_val)[:, 1], xgb_scores)
print("saved file reloads and gives the same scores:", same)
print(round(os.path.getsize("../models/fraud_model.joblib") / 1e6, 1), "MB")

saved file reloads and gives the same scores: True
1.2 MB
